# Detecção de Objetos

## Estrutura dos Dados

Quando trabalharmos com um projeto de visão computacional para detecção de objetos com algoritmos como YOLO, os dados devem ser estruturados contendo as imagens e um arquivo txt referente a cada arquivo de imagem.

Um modelo de classificação entrega uma única informação: a classe predita. Já os modelos de detecção de objetos retorna mais do que isso. Além da classe, também retorna mais 4 números entre 0 e 1 que representam a caixa ou as caixas em que o objeto se encontra na imagem.

Neste projeto, para os conjuntos de treino, validação e teste temos duas subpastas: images e labels. A pasta images abriga os arquivos de imagem e a pasta labels abriga os arquivos txt referentes a elas. Para cada imagem com um nome de arquivo, deve haver um arquivo de texto com o mesmo nome informando os dados esperados como resultado.

```shell
├── data
│   ├── test
│   │   ├── images
│   │   └── labels
│   ├── train
│   │   ├── images
│   │   └── labels
│   └── val
│       ├── images
│       └── labels
├── src
│   └── detection
├── data.yaml
└── ...
``

Eis um exemplo de arquivo de texto. Cada linha representa uma ocorrência do objeto na imagem. 

```txt
1 0.35833333333333334 0.4291666666666667 0.12083333333333333 0.04166666666666667
1 0.553125 0.42638888888888893 0.08541666666666667 0.06944444444444445
0 0.68125 0.24305555555555555 0.5708333333333333 0.4361111111111111
```

Os dados são distribuídos como no esquema abaixo. Com base nesses números, conseguimos calcular a localização esperada das caixas. Sendo ela relativa ao tamanho da imagem. Utilizando esses dados, já somos capazes de determinar de forma precisa a localização das caixas.

```text
<classe> <x_centro> <y_centro> <largura> <altura>
```

### Arquivo de Configuração

Na raíz no projeto, teremos um arquivo data.yaml. Nele, definimos informações sobre os dados. Por exemplo, a propriedade path guarda um caminho relativo base para os dados. Nesse caso, o diretório data. As propriedades train, val e test apontam para o caminho relativo do diretório de imagens desses conjuntos de dados, considerando como base o path. 

A propriedade names poderá receber uma espécie de dicionário com ```chave: valor```. Essa é a definição das classes e seus nomes. A chave é o valor que existe nos arquivos de texto. Nesse projeto, 0 e 1. Já os valores devem ser o nome da classe em texto. Nesse caso, "Fumaça" ou "Fogo".

Nesse arquivo, podemos definir outras propriedades como nc (Número de classes), train_count (Número de imagens do conjunto de treino), val_count e test_count.

```yaml
path: ./data
train: train/images
val: val/images
test: test/images

names:
  0: Fumaça
  1: Fogo

nc: 2
train_count: 14122
val_count: 3099
test_count: 4306
```

## Métricas de Precisão

Na detecção de objetos, temos dois critérios de sucesso: a classificação correta e a localização correta. A novidade aqui é a localização. Para ela, há uma métrica chamada IoU (Intersection over Union) ou Interseção sobre União. Imageme que em um plano temos dois retângulos. A união deles é a área total que os dois ocupam juntos ou separadamente. A interseção é o quanto de área os dois ocupam juntos. A métrica IoU é a razão entre o quanto de área os dois ocupam juntos e o quanto de área os dois ocupam juntos ou separadamente. Um IoU = 1.0 indica sobreposição perfeita. Maior ou igual a 0.50 é o padrão histórico aceito para considerar uma detecção correta e maior ou igual a 0.75 é considerada uma detecção de alta precisão métrica.

### Métrica mAP

As métricas mAP (Mean Average Precision) avaliam a acurácia do modelo de acordo com os valores IoU. A métrica mAP@50 por exemplo informa qual percentual das imagens atingiu IoU médio maior ou igual a 0.50. Outra métrica mAP@50-95 calcula os percentuais para IoU 0.50, 0.55, 0.60, ..., 0.95 e tira uma média dos percentuais. Para essa métrica, valores acima de 0.40 (40%) são considerados excelentes.

## Escolha do Modelo

O modelo que utilizaremos para detecção de objetos é o YOLO (You Only Look Once). Existem diversas variantes e pesos para esse modelo. Confira a tabela abaixo.

| Variante | Pesos | Parâmetros | Tamanho | Latência | Cenário Ideal |
| -------- | ----- | ---------- | ------- | -------- | ------------- |
| Nano (n) | yolov8n.pt, yolo11n.pt | ~3.2M | ~6 MB | ~1.5 ms | Dispositivos de borda como Raspberry Pi e celulares. Viável para rodar em CPU ou para prototipagem básica |
| Small (s) | yolov8s.pt, yolo11s.pt | ~11.2M | ~22 MB | ~2.5 ms | Equilíbrio entre acurácia e performance |
| Medium (m) | yolov8m.pt, yolo11m.pt | ~25.9M | ~50 MB | ~4.7 ms | Tarefas complexas com dezenas de classes e sobreposição visual |
| Large (l) | yolov8l.pt, yolo11l.pt | ~43.7M | ~87 MB | ~7.5 ms | Quando a GPU de inferência  potente e a precisão é prioritária |
| XLarge (x) | yolov8x.pt, yolo11x.pt | ~68.2M | ~136 MB | ~11.0 ms | Competições acadêmicas e benchmarks de pesquisa |

## Transfer Learning

Ao trabalhar com detecção de objetos, na esmagadora maioria dos casos, estaremos fazendo transfer learning em modelos pré-treinados. O YOLO foi treinado de diversas formas diferentes com o propósito de poder ser utilizado em vários ambientes, como explicitado na tabela anterior. 

Para treinar um modelo, começamos instanciando ele utilizando a classe YOLO da biblioteca ultralytics. A classe recebe como primeiro parâmetro o identificador dos pesos que queremos utilizar inicialmente. Nesse caso, estamos utilizando os pesos yolov8s.pt que é referente à variante pequena do modelo, equilibrando performance e acurácia.

Após instanciar o modelo, chamamos o método train, passando uma série de parâmetros. Confira o exemplo para saber ao quê cada um se refere.

In [ ]:
from ultralytics import YOLO

def treinar_detector():
    model = YOLO("yolov8s.pt")

    results = model.train(
        data="data.yaml",        # Diretório do arquivo de configuração dos dados
        epochs=60,               # Número de épocas do treinamento completo
        imgsz=640,               # Tamanho de redimensionamento das imagens para treino
        batch=16,                # Número de imagens por lote
        patience=15,             # Interrompe o treinamento de o mAP50-95 não melhorar por 15 épocas consecutivas
        workers=4,               # Subprocessos para carregamento das imagens
        device=0,                # Dispositivo para o treinamento (0 = cuda) ou "cpu" para processador
        save=True,               # Salva automaticamente pesos periódicos
        save_period=10,          # Salva checkpoints de pesos a cada 10 épocas
        project="experimentos",  # Nome da pasta raíz dos resultados
        name="detector_v1",      # Subpasta do experimento atual
        optimizer="auto",        # Ajusta automaticamente entre SGD e AdamW
        verbose=True
    )
    
    print("Treinamento finalizado com sucesso!")
    print(f"Melhores pesos salvos em: {results.save_dir}/weights/best.pt")

if __name__ == "__main__":
    treinar_detector()

### Como o Modelo é Treinado?

#### Troca da Cabeça de Decisão

Quando trabalhamos com modelos de classificação como ResNet e MobileNet, realizamos a mesma técnica de transfer learning que estamos fazendo no YOLO. Apenas o método funciona de forma um pouco diferente. Enquanto no ResNet congelávamos os gradientes e rodávamos algumas épocas apenas para calibrar a cabeça, aqui, são executadas algumas épocas (warmup_epochs) com uma taxa de aprendizado muito pequena (ex: 0.0001) e essa taxa sobe em linha reta conforme as épocas avançam. Isso permite que a cabeça seja calibrada sem descompensar o restante do corpo do modelo.

Mas, é possível fazer o transfer learning exatamente igual. Basta utilizar o parâmetro freeze que definirá o número de camadas que serão congeladas. 10 camadas congelarão todo o backbone extrator de bordas. 22 camadas congelarão o backbone completo e deixarão apenas a cabeça destravada. Para conjuntos de dados pequenos, pode ser ideal utilizar esse parâmetro para congelar algumas camadas (ex.: 10 camadas).

#### Ciclo de Treinamento

Bom, agora que o corpo do modelo está adaptado e os parâmetros estão todos definidos e evoluindo de maneira ótima, é hora de iniciar o treinamento.

O YOLO inicialmente divide as imagens utilizando malhas em três escalas (strides 8, 16 e 32). Em uma imagem de 640x640, teremos 80x80 células na malha pequena (para objetos minúsculos), 40x40 na média e 20x20 na grande, totalizando **8.400 palpites** de caixas por imagem.

Mas como o YOLO escolhe quais dessas caixas utilizar para aprender? Inicialmente, são filtradas apenas as caixas cujo centro cai dentro do retângulo do objeto real, descartando a enorme maioria dos palpites irrelevantes. Em seguida, o algoritmo **Task-Aligned Assigner (TAL)** calcula uma métrica de alinhamento que combina a certeza da classe predita com a sobreposição espacial (IoU). As caixas que melhor "vestem" o tamanho real do objeto e acertam a classe ganham as maiores notas.

Dentre elas, são selecionadas por padrão as **10 melhores caixas** para servirem de exemplos positivos. É aplicado um sistema de alvos suaves (*soft targets*) em que a 1ª colocada se torna a referência mais forte de aprendizado, e as seguintes ganham pesos proporcionalmente menores. Todo o restante das caixas é marcado como fundo (*background*): a perda de coordenadas delas é desligada e a perda de classe é treinada para zero.

A função de perda calcula simultaneamente três perdas: a **CIoU** (sobreposição, centro e formato da caixa), a **BCE** (classificação probabilística das classes) e a **DFL** (Distribution Focal Loss), que prevê distribuições contínuas de probabilidade para as bordas — sendo o grande diferencial para detectar fumaça, que é semitransparente e sem margem rígida.

#### Otimização do Modelo

A cada lote (*batch*) de imagens, os algoritmos de otimização calculam os gradientes e ajustam os pesos da rede na direção contrária ao erro. Mas há um cuidado fundamental: durante o treino, alguns lotes específicos podem conter imagens atípicas ou ruídos que causariam solavancos bruscos nos pesos.

Por isso, o YOLO mantém dois modelos em memória: o modelo ativo (que sofre os ajustes a cada lote) e o **modelo sombra (EMA - Exponential Moving Average)**. A cada lote, os pesos do modelo sombra são atualizados atribuindo ~99.99% de importância ao histórico acumulado e apenas ~0.01% à novidade do lote atual. Dessa forma, o modelo sombra filtra as oscilações e mantém uma evolução limpa e estável.

Ao final de cada época, o modelo sombra é testado no conjunto de validação. Se ele superar o recorde anterior de mAP, ele é salvo como **`best.pt`**.

## Artefatos de Validação

Ao executar o treinamento, uma pasta runs será criada na raíz do projeto. Sob algumas camadas de pastas, para cada execução do treinamento, deve haver um conjunto de pastas e arquivos que são os artefatos de validação de uma execução de treinamento.

```shell
├── runs
│   └── detect
│       └── experimentos        # Execuções do treinamento
│           ├── detector_v1
│           └── detector_v1-2
└── ...
```

Dentro da pasta de execução de um treinamento, existe a pasta weights que carrega arquivos conforme o esquema abaixo.

```shell
├── ...
│   └── weights
│       ├── best.pt          # Pesos da época com o maior mAP50-95
│       ├── epoch0.pt        # Checkpoint a cada X épocas
│       ├── epoch10.pt       # Checkpoint a cada X épocas
│       ├── epoch20.pt       # Checkpoint a cada X épocas
│       ├── epoch30.pt       # Checkpoint a cada X épocas
│       ├── epoch40.pt       # Checkpoint a cada X épocas
│       ├── epoch50.pt       # Checkpoint a cada X épocas
│       └── last.pt          # Pesos da última época do treinamento
└── ...
```

Além deles, há arquivos visuais que nos ajudam a entender e visualizar métricas do modelo treinado.

```shell
├── ...
│   ├── weights
│   ├── val_batch*_pred.jpg                 # Imagens da validação com caixas desenhadas
│   ├── BoxF1_curve.png                     # Curva entre confiança e acurácia
│   ├── confusion_matrix_normalized.png     # Matriz de confusão normalizada
│   └── results.png                         # Gráficos de progressão do treinamento
└── ...
```

### Imagens de Validação com Caixas Desenhadas

A amostra de imagem abaixo contém todas as imagens de um lote. Essas imagens foram processadas pelo modelo e foram devolvidas com as caixas e classes preditas desenhados. Esses arquivos servem para verificarmos visualmente o que o modelo está fazendo.

![Figura 1](assets/images/figura_01.jpg)

### Curva de Confiança e Acurácia

Esta imagem é um gráfico de linha que revela a relação entre a confiança esperada e acurácia. Para uma confiança esperada de 0.5, por exemplo, o modelo alcançou uma pontuação F1 de aproximadamente 0.70 (70%).

![Figura 2](assets/images/figura_02.png)

### Matriz de Confusão Normalizada

A matriz de confusão normalizada é o que nos dirá como se distribuíram as classificações do modelo. No eixo X há os valores reais e no eixo Y os valores preditos. Onde a classe colide com si própria é onde o modelo acertou. Nesse caso, acertou 96% das ocorrências de fogo e 94% das ocorrências de fumaça corretamente. A coluna de dados reais para background essencialmente aponta qual das classes o modelo viu mais onde não tinha nada. Nesse caso, em 59% ele inventou fogo onde não tinha. Mas parece uma informação assustadora, não é? Mas isso acontece porque para gerar o gráfico de confiança e precisão, é preciso executar a validação com 0.001 de confiança por exemplo. Dessa forma, qualquer mínimo sinal é classificado entre as classes, mesmo que seja apenas o fundo da imagem.

![Figura 3](assets/images/figura_03.png)

### Progressão do Treinamento

Os gráficos de progressão do treinamento desenham uma linha através de todas as épocas do treinamento, revelando como uma determinada métrica evoluiu durante a sessão completa. Idealmente, as métricas box_loss, cls_loss e dfl_loss tem de andar grosseiramente juntas. Imagine que ao mesmo tempo que a perda de treinamento aumenta, a perda da validação diminui. Isso pode indicar que o modelo não está conseguindo extrair sinais valiosos da imagem, e sim "decorando" o conjunto de dados. Há também outros gráficos que mostram como evoluiram as métricas de acurácia, recall, mAP50 e mAP50-95.

![Figura 4](assets/images/figura_04.png)

## Inferência

Em um arquivo infer.py, instanciamos o modelo YOLO passando como parâmetro um caminho até os pesos salvos no treinamento no arquivo best.pt. Para realizazr uma inferência utilizando o modelo, podemos chamar o método predict dele. Este método receberá como parâmetro source a origem dos dados a serem processados. Nesse caso, uma pasta input na raíz do projeto. Outros parâmetros como conf definem a confiança necessária para marcar um objeto na imagem, IoU define a sobreposição mínima e save define se os dados processados devem ser salvos ou não. Quando sim, pode-se definir os parâmetros project que determina uma pasta primária para armazenar os dados processados e name que determina uma subpasta. 

O único parâmetro obrigatório de fato é o parâmetro source. Todos os outros assumem valores padrão.

In [ ]:
from ultralytics import YOLO

from detection.config import INPUT_DIR, WEIGHTS_PATH

model = YOLO(WEIGHTS_PATH)

results = model.predict(
    source=INPUT_DIR,
    conf=0.35,
    iou=0.50,
    project="output",
    name="predictions",
    save=True,
)

# ...


O método predict retorna um objeto de resultados que guarda as informações sobre as inferências para cada imagem da origem. Podemos iterar os resultados em um laço de repetição, como abaixo. Cada resultado terá uma lista de caixas na propriedade boxes e também um dicionário com os nomes das classes em uma propriedade names. Cada caixa tem uma propriedade cls que é um tensor que contém o resultado da classe predita para aquela caixa. Além disso, as caixas também tem as propriedades conf que informa o grau de confiança da detecção e algumas propriedades de localização da caixa como xyxy, que é um tensor que contém as coordenadas X de início e final e as coordenadas Y de início e final da caixa.

In [ ]:
# ...

for result in results:
    for box in result.boxes:
        class_id = int(box.cls[0])
        class_name = result.names[class_id]

        confidence = float(box.conf[0])

        x1, y1, x2, y2 = box.xyxy[0].int().tolist()

        print(f"""
-> {class_name}
   Confiança: {confidence:.2%}
   Dimensões: {x2 - x1}x{y2 - y1}
   Coordenadas: {x1, y1, x2, y2}
""")


**Imagem Original de Exemplo**

![Figura 5](assets/images/figura_05.jpg)

**Resultado com Caixas e Classes**

![Figura 6](assets/images/figura_06.jpg)

**Saída no Terminal**

```shell
-> Fumaça
   Confiança: 81.71%
   Dimensões: 1110x677
   Coordenadas: (391, 57, 1501, 734)


-> Fogo
   Confiança: 39.82%
   Dimensões: 154x83
   Coordenadas: (1297, 614, 1451, 697)
```

O resultado da predição e as caixas têm muitas propriedades e informações úteis. Encorajamos que esses objetos sejam explorados.

## Recortando Evidências

Ao realizar uma predição, podemos selecionar um pedaço da imagem original. A propriedade orig_img do resultado contém esta. Para recortá-la, podemos utilizar síntaxe de array especificando os intervalos de coordenadas Y e X, em ordem. Para salvar este recorte de imagem, podemos utilizar o método imwrite do cv2. Passando como parâmetro o destino e a imagem recortada. A propriedade path do resultado armazena a localização da imagem original.

In [ ]:
# ...

from pathlib import Path

import cv2

for result in results:
    for box_index, box in enumerate(result.boxes):
        x1, y1, x2, y2 = box.xyxy[0].int().tolist()

        box_img = result.orig_img[y1:y2, x1:x2]

        cv2.imwrite(
            f"output/evidencias/evidencia_{Path(result.path).stem}_{box_index}.jpg", box_img
        )


## Processando Vídeos

Existem várias formas de processar vídeos, a primeira delas é utilizando uma pasta contendo os vídeos e passando como parâmetro source. Outra forma de receber vídeo é utilizando a webcam. Para isso, passamos o número zero como source. Isso definirá que a origem do vídeo é o dispositivo primário de entrada de vídeo do dispositivo. Por padrão, cada frame do vídeo será armazenado em memória. O parâmetro stream evita que isso aconteça, possibilitando ainda tratarmos cada frame. Para conseguirmos visualizar o vídeo processado, podemos utilizar o parâmetro show. Por fim, o parâmetro verbose determina se a saída no terminal deve ser muito detalhada ou não.

In [ ]:
from ultralytics import YOLO

from detection.config import WEIGHTS_PATH

model = YOLO(WEIGHTS_PATH)

results = model.predict(
    source=0,
    stream=True,
    show=True,
    verbose=False,
)

# ...

Iterando o resultado da predição em vídeo, processamos os frames um a um como imagens individuais, cada uma podendo conter caixas de detecção. Nesse projeto que estamos desenvolvendo, precisamos detectar quando há um incêndio. Portanto, procuramos nos frames do vídeo caixas que identifiquem fogo.

In [ ]:
# ...

for result in results:
    for box_index, box in enumerate(result.boxes):
        if result.names[int(box.cls[0])] == "Fogo":
            print("Incêndio detectado!")

### Persistência Temporal

Se rodarmos o código como está e fogo aparecer na câmera, o script avisará pelo terminal. Mas, avisará mesmo que um único frame contenha a caixa identificada. O risco de alarme falso é iminente. Então precisamos de alguma forma que possa garantir com mais confiança que um incêndio foi detectado.

Para passar por cima desse problema, podemos utilizar um deque. Um deque é como uma lista, mas tem um limite de tamanho e quando um item é adicionado ao final, se ultrapassar o limite de tamanho, o item mais antigo será colocado para fora. É ideal para esse tipo de aplicação. Nesse projeto, criamos um deque com tamanho máximo de 60 itens. A cada frame processado, adicionamos ao deque um booleano que indica se o frame continha uma caixa identificadora de fogo. Para cada frame, também verificamos a soma de todos os itens do deque. Se a soma for maior ou igual a 40 (soma de booleanos 0 ou 1), então dizemos que há fogo. Dessa forma, um único frame conter a caixa de fogo não é suficiente. Dentre os últimos 60 frames, ao menos 40 deles tem de conter a caixa identificadora de fogo para dizermos que de fato há um incêndio, evitando falsos alarmes. Por padrão, são processados 30 frames por segundo. Portanto, 60 frames representam 2 segundos.

In [ ]:
# ...

from collections import deque

buffer = deque(maxlen=60)

has_fire = False

for result in model.predict(
    source=0,
    stream=True,
    show=True,
    verbose=False,
):
    frame_has_fire = False

    for box_index, box in enumerate(result.boxes):
        if result.names[int(box.cls[0])] == "Fogo":
            frame_has_fire = True

            break

    buffer.append(frame_has_fire)

    if sum(buffer) >= 40:
        has_fire = True

        print("Incêndio detectado!")
    else:
        has_fire = False

## Exportação do Modelo

Na vida real, nem sempre o modelo de detecção será executado por uma máquina potente com placa de vídeo dedicada e muitos gigabytes de RAM. 

O conceito de Edge Computing é relativo aos dispositivos finais que de fato executarão o nosso modelo. Denominamos estes dispositivos como dispositivos de borda. Sendo por exemplo um Raspberry Pi, celular, tablet, etc. 

Para possibilitar que o modelo seja integrado a esses dispositivos, utilizamos o formato ONNX (Open Neural Network Exchange) que praticamente qualquer linguagem de programação consegue interpretar e executar.

Para exportar o modelo nesse formato, basta utilizarmos o método export do modelo treinado. Esse método poderá receber algumas definições como o parâmetro format que define o formato do arquivo de exportação (onnx) e imgsz que define o tamanho das imagens ou frames que serão processados. O parâmetro dynamic define se o tamanho dos lotes deve ser dinâmico. O parâmetro simplify define se operadores redundantes do grafo serão simplificados. É muito importante que sejam, a fim de otimizar ainda mais o modelo exportado deixando de lado operações redundantes. O método export retornará um caminho que informa onde o arquivo exportado foi salvo.

In [ ]:
from ultralytics import YOLO

from detection.config import IMAGE_SIZE, WEIGHTS_PATH

model = YOLO(WEIGHTS_PATH)

onnx_path = model.export(
    format="onnx",
    imgsz=IMAGE_SIZE,
    dynamic=True,
    simplify=True
)

print(f"Modelo exportado para: {onnx_path}")

Para utilizar o modelo exportado, podemos utilizar a classe YOLO do próprio Ultralytics que aceita o arquivo de pesos em formato ONNX. Mas, em máquinas mais modestas, o ideal é utilizar a biblioteca onnxruntime junto ao opencv. É uma abordagem mais manual e trabalhosa, mas é necessária para dispositivos de borda.